# Code for Calculating the Frobenius Norm of target unitary from different GRAPE functions

In this notebook, we use two different functions from the qutip control library ("optimize_pulse_unitary" from the pulseoptim object and "cy_grape_unitary" from the grape object), in order to implement the GRAPE algorithm. From this we obtain a final Unitary $\hat U$ which should be close to some given target Unitary $\hat U^*$. We then calculate the Frobenius norm squared of the difference between these two unitary as a measure of the error fidelity:

$$ || A  ||_{F}^2  = Tr(A^{\dagger}A) = \sum_{ij}^{N} |A_{ij}|^{2}$$


These values are then plotted against previously obtained results which obtained the unitary by reformulating the system in terms of a polynomial equation (https://arxiv.org/pdf/2209.05790.pdf) 

## Imports

Below are the necessary imports for running this code:

1) Qutip functions: 


The first line does import all qutip functions from the base object, which is used here to convery numpy arrays to Quantum objects using Qobj() and qeye() to create an identity operator of a specified size (here 3) as the initial starting point for the GRAPE algorithm.

The next set of imports are functions used for the pulseoptim class and a logger class which formats the result when being printed to terminal/ saving the to a file (I don't think we need them)

The final set of qutip imports are for implementing the cy_grape method. The TextProgressBar is justed used as a convenient way of tracking the progress of the function. (plot_grape_control_fields and _overlap are not necessary any more but are just used as tools for plotting the grape control fields and calculating the trace norm respectively)

2) Other imports:


The next set of imports are just the basic imports for simple tasks: 
matplotlib for plotting. 

numpy for storing and manipulating data.

h5py for reading and writing hdf5 files. 

time for time-keeping.  


In [19]:
from qutip import *

#QuTiP control modules
import qutip.control.pulseoptim as cpo
#import qutip.logging_utils as logging
#logger = logging.get_logger()
#Set this to None or logging.WARN for 'quiet' execution
#log_level = logging.INFO

from qutip.control import * 
from qutip.ui.progressbar import TextProgressBar
from qutip.control.grape import plot_grape_control_fields, _overlap


%matplotlib inline
import numpy as np
from numpy import linalg as LA
import matplotlib.pyplot as plt
#import datetime
import h5py
import time 
start_time = time.time()

from tqdm.notebook import tqdm

from multiprocessing import Pool

## Preparing the input data
 
 

 
In the cell below we read data from hdf5 files. The first is a list of 1000 3x3 target unitaries and the second is the frobenius norm calculated within the aforementioned paper. 

*Note: Bottom half of this cell may change as appending to an array can be computationally expensive* 

Sample here is used to denote the number of unitaries from the list that we would like to select and a for-loop is used to convert these unitaries into Qobj type which are stored in an array. Two empyt lists are also initialised which are used for storing data. 


In [20]:
# Read the fixed targets generated by generate_targets.jl
target_results_file = "targets.hdf5"

# The POP results to compare against
pop_results_m = 2
pop_results_file = f"results_{pop_results_m}.hdf5"

# Compare CRAB/GRAPE over these piecewise time-slice counts.
timeslices = [2, 4, 6, 8]

def read_text_dataset(h5file, key):
    if key not in h5file:
        return []
    raw = h5file[key][()]
    if isinstance(raw, bytes):
        return raw.decode().splitlines()
    if isinstance(raw, str):
        return raw.splitlines()
    return [_.decode() if isinstance(_, bytes) else str(_) for _ in raw]

with h5py.File(target_results_file, 'r') as resultsFile:
    raw_targets = resultsFile["U_targets"][...]
    # Julia/HDF5 arrays are seen by h5py as (3, 3, n_samples).
    U_targets = raw_targets.transpose(2, 1, 0) if raw_targets.shape[0] == 3 else raw_targets
    target_family = read_text_dataset(resultsFile, "target_family")
    target_random_strength = resultsFile["target_random_strength"][()] if "target_random_strength" in resultsFile else np.nan

# Load POP results for baseline comparison
with h5py.File(pop_results_file, 'r') as popFile:
    pop_norm_U_target_minus_obtained = popFile["norm_U_target_minus_obtained"][...]
    pop_fidelity = popFile["f_PSU"][...]
    pop_fid_error = 1 - pop_fidelity
    
sample = U_targets.shape[0]
if not target_family:
    target_family = [f"target_{i}" for i in range(sample)]


## Choosing input parameters

The dynamics of the system are generated by the time dependent Hamiltionian:

$$H(t) = H_{0} + \sum_{i} u_i(t)H_{i} $$

Where $H_{0}$ is the drift Hamiltonian, $H_{i}$ are the control Hamiltonians and $u_{i}$ are the control fields. 

In the cell below we create Qobjs for $H_{0}$ and the single control $H_{c}$ according to the previously mentioned paper. We choose the Identity operator as the starting point for the GRAPE algorithm. We also can set the period over which the algorithm runs and the number of time slots at which act as the number of steps taken find a minimum in the landscape. 

(See https://qutip.org/docs/latest/guide/guide-control.html and the paper above for a more detail discussion).

In [21]:
H_drift_matrix = np.array(
    [[0, 0, 0],
     [0, 3.21505101e+10, 0],
     [0, 0, 6.23173079e+10]]
)
H_drift_matrix /= H_drift_matrix.max()

H_control_matrix = np.array(
    [[0, 1, 0],
     [1, 0, 1.41421356],
     [0, 1.41421356, 0]]
)
H_control_matrix /= H_control_matrix.max()

H_drift = Qobj(H_drift_matrix)
H_control = [Qobj(H_control_matrix)] 
 
# Unitary starting point
U_0 = qeye(3)

# Time allowed for the evolution
evo_time = 0.5

## pulseoptim

In the two cells below we run the function from the pulseoptim subclass. The first cell we select some more parameters for this function, these are:
 1) fid_error_tar - when this fidelity error is achieved we can terminate the algorithm as we have sufficiently achieved our goal.
 2) max_iterations - the max number of grape iterations 
 3) max_wall_time - maximum time the algorithm runs before terminating if the target unitary has not been reached
 4) min_grad - minimum gradient to determine whether we are trapped in a local minima rather than a global minima 
 5) p_type - the type of pulse for the control fields (here set to random)
 
 
 The second cell then runs the algorithm with these parameters for each unitary in the sample size. The finial unitary is stored in the "U_final_pulseoptim" list and for each iteration the termination reason and the fidelity error is printed out. You can also extract other useful information from the result variable such as the final control fields, the number of iterations used etc. 


In [22]:
#pulse optim extra params 

# Fidelity error target
fid_err_targ = 1e-10
# Maximum iterations for the optisation algorithm
max_iter = 1000
# Maximum (elapsed) time allowed in seconds
max_wall_time = 1000
# Minimum gradient (sum of gradients squared)
# as this tends to 0 -> local minima has been found
min_grad = 1e-10

# pulse type alternatives: RND|ZERO|LIN|SINE|SQUARE|SAW|TRIANGLE|
p_type = 'RND'

# Control amplitude bounds (must match POP)
control_bound = 1.0 

In [23]:
def optimize(U_tag, alg, n_ts):
    return cpo.optimize_pulse_unitary(
        H_drift, H_control, U_0, Qobj(U_tag), n_ts, evo_time,
        fid_err_targ=fid_err_targ, min_grad=min_grad,
        max_iter=max_iter, max_wall_time=max_wall_time,
        # log_level=log_level,
        init_pulse_type=p_type,
        alg=alg,
        amp_lbound=-control_bound,
        amp_ubound=control_bound,
        # gen_stats=True
    )

def optimize_timed(U_tag, alg, n_ts):
    start = time.time()
    result = optimize(U_tag, alg, n_ts)
    return result, time.time() - start

In [24]:
qutip_results_by_timeslice = {}

for n_ts in timeslices:
    print(f"Running CRAB/GRAPE for n_ts={n_ts}")

    crab_runs = [
        optimize_timed(U_targets[i, :, :], alg='CRAB', n_ts=n_ts)
        for i in tqdm(range(sample), desc=f"CRAB n_ts={n_ts}")
    ]
    grape_runs = [
        optimize_timed(U_targets[i, :, :], alg='GRAPE', n_ts=n_ts)
        for i in tqdm(range(sample), desc=f"GRAPE n_ts={n_ts}")
    ]

    crab_results = [run[0] for run in crab_runs]
    grape_results = [run[0] for run in grape_runs]
    U_final_pulseoptim = [result.evo_full_final for result in crab_results]
    U_final_cyGRAPE = [result.evo_full_final for result in grape_results]

    pulseoptim_fidelity = np.zeros(sample)
    cy_grape_fidelity = np.zeros(sample)
    for i in range(sample):
        pulseoptim_fidelity[i] = abs(_overlap(Qobj(U_targets[i, :, :]), U_final_pulseoptim[i])) ** 2
        cy_grape_fidelity[i] = abs(_overlap(Qobj(U_targets[i, :, :]), U_final_cyGRAPE[i])) ** 2

    qutip_results_by_timeslice[n_ts] = {
        "pulseoptim_fidelity": pulseoptim_fidelity,
        "cy_grape_fidelity": cy_grape_fidelity,
        "pulseoptim_fid_error": 1 - pulseoptim_fidelity,
        "cy_grape_fid_error": 1 - cy_grape_fidelity,
        "crab_optimizer_fid_error": np.array([result.fid_err for result in crab_results], dtype=float),
        "grape_optimizer_fid_error": np.array([result.fid_err for result in grape_results], dtype=float),
        "crab_elapsed_time": np.array([run[1] for run in crab_runs], dtype=float),
        "grape_elapsed_time": np.array([run[1] for run in grape_runs], dtype=float),
        "crab_termination_reason": [result.termination_reason for result in crab_results],
        "grape_termination_reason": [result.termination_reason for result in grape_results],
    }

    print(
        f"n_ts={n_ts}: median CRAB infidelity={np.median(qutip_results_by_timeslice[n_ts]['pulseoptim_fid_error']):.3e}, "
        f"median GRAPE infidelity={np.median(qutip_results_by_timeslice[n_ts]['cy_grape_fid_error']):.3e}"
    )

Running CRAB/GRAPE for n_ts=2


CRAB n_ts=2:   0%|          | 0/100 [00:00<?, ?it/s]

GRAPE n_ts=2:   0%|          | 0/100 [00:00<?, ?it/s]

n_ts=2: median CRAB infidelity=1.191e-03, median GRAPE infidelity=1.087e-03
Running CRAB/GRAPE for n_ts=4


CRAB n_ts=4:   0%|          | 0/100 [00:00<?, ?it/s]

...

GRAPE n_ts=6:   0%|          | 0/100 [00:00<?, ?it/s]

n_ts=6: median CRAB infidelity=1.143e-03, median GRAPE infidelity=1.019e-03
Running CRAB/GRAPE for n_ts=8


CRAB n_ts=8:   0%|          | 0/100 [00:00<?, ?it/s]

GRAPE n_ts=8:   0%|          | 0/100 [00:00<?, ?it/s]

n_ts=8: median CRAB infidelity=1.194e-03, median GRAPE infidelity=1.017e-03


In [25]:
# CRAB and GRAPE are both run in the time-slice sweep above.


for i in tqdm(range(sample)):
    result = cpo.optimize_pulse_unitary(
                H_drift, H_control, U_0, Qobj(U_targets[i, :, :]), n_ts, evo_time, 
                fid_err_targ=fid_err_targ, min_grad=min_grad, 
                max_iter=max_iter, max_wall_time=max_wall_time, 
                # log_level=log_level,
                init_pulse_type=p_type, 
                #alg='GRAPE',
                alg='CRAB',
                gen_stats=False
            )
    print("Final fidelity error {}".format(result.fid_err))
    print("Terminated due to {}".format(result.termination_reason))
   
    #Here all terminate due to convergence however if it terminates due to a different reason may cause a problem
    #U_final_pulseoptim.append(result.evo_full_final)
    
    break

## cy_grape 

In these two cells below we set the the parameters and run the cy_grape_unitary function. There are less inputs in this method:

1) times - an array of where the number of elements = number of time slots and the final element is the period.
2) n_iterations - number of grape iterations

the second cell follows the same procedure as the cell above, however in this case we can track the progress of each iteration with TextProgressBar(). Here we only the store the final unitary, however one could extract other information from the result variable such as the final control fields (for each grape iteration). 


## Plotting Fidelity error (trace norm)
This code is for calculating the trace norm (overlap) of the obtained and target unitaries:

$\frac{1}{d} |Tr[\hat U^{\dagger}\hat U^{*}]| $,

$d$ is the dimension of $\hat U$.

(currently not plotting this just calculating it. The cells for plotting have been commented out and folded as they still use TSSOS but will modify if needed)




In [26]:
print("n_ts, POP median, CRAB median, GRAPE median")
for n_ts in timeslices:
    result = qutip_results_by_timeslice[n_ts]
    print(
        f"{n_ts}, "
        f"{np.median(pop_fid_error):.3e}, "
        f"{np.median(result['pulseoptim_fid_error']):.3e}, "
        f"{np.median(result['cy_grape_fid_error']):.3e}"
    )

pulseoptim_fid_error_by_timeslice = np.vstack([
    qutip_results_by_timeslice[n_ts]["pulseoptim_fid_error"] for n_ts in timeslices
])
cy_grape_fid_error_by_timeslice = np.vstack([
    qutip_results_by_timeslice[n_ts]["cy_grape_fid_error"] for n_ts in timeslices
])
crab_elapsed_time_by_timeslice = np.vstack([
    qutip_results_by_timeslice[n_ts]["crab_elapsed_time"] for n_ts in timeslices
])
grape_elapsed_time_by_timeslice = np.vstack([
    qutip_results_by_timeslice[n_ts]["grape_elapsed_time"] for n_ts in timeslices
])


n_ts, POP median, CRAB median, GRAPE median
2, 1.088e-03, 1.191e-03, 1.087e-03
4, 1.088e-03, 1.086e-03, 1.084e-03
6, 1.088e-03, 1.143e-03, 1.019e-03
8, 1.088e-03, 1.194e-03, 1.017e-03


# Writing data to hdf5

In [27]:
for n_ts in timeslices:
    result = qutip_results_by_timeslice[n_ts]
    with h5py.File('RND_qutip_opt_results_timeslices_' + str(n_ts) + '.hdf5', 'w') as hf:
        hf.create_dataset("target_family", data="\n".join(target_family))
        hf.create_dataset("target_random_strength", data=target_random_strength)
        hf.create_dataset("source_pop_file", data=target_results_file)
        hf.create_dataset("U_targets", data=U_targets)
        hf.create_dataset("pop_fid_error", data=pop_fid_error)
        hf.create_dataset("pop_fidelity", data=pop_fidelity)
        hf.create_dataset("pop_norm_U_target_minus_obtained", data=pop_norm_U_target_minus_obtained)
        hf.create_dataset("pulseoptim_fid_error", data=result["pulseoptim_fid_error"])
        hf.create_dataset("cy_grape_fid_error", data=result["cy_grape_fid_error"])
        hf.create_dataset("pulseoptim_fidelity", data=result["pulseoptim_fidelity"])
        hf.create_dataset("cy_grape_fidelity", data=result["cy_grape_fidelity"])
        hf.create_dataset("crab_optimizer_fid_error", data=result["crab_optimizer_fid_error"])
        hf.create_dataset("grape_optimizer_fid_error", data=result["grape_optimizer_fid_error"])
        hf.create_dataset("crab_elapsed_time", data=result["crab_elapsed_time"])
        hf.create_dataset("grape_elapsed_time", data=result["grape_elapsed_time"])
        hf.create_dataset("crab_termination_reason", data="\n".join(result["crab_termination_reason"]))
        hf.create_dataset("grape_termination_reason", data="\n".join(result["grape_termination_reason"]))

with h5py.File('RND_qutip_opt_results_timeslices_summary.hdf5', 'w') as hf:
    hf.create_dataset("timeslices", data=np.array(timeslices, dtype=int))
    hf.create_dataset("target_family", data="\n".join(target_family))
    hf.create_dataset("target_random_strength", data=target_random_strength)
    hf.create_dataset("source_pop_file", data=target_results_file)
    hf.create_dataset("U_targets", data=U_targets)
    hf.create_dataset("pop_fid_error", data=pop_fid_error)
    hf.create_dataset("pop_fidelity", data=pop_fidelity)
    hf.create_dataset("pop_norm_U_target_minus_obtained", data=pop_norm_U_target_minus_obtained)
    hf.create_dataset("pulseoptim_fid_error_by_timeslice", data=pulseoptim_fid_error_by_timeslice)
    hf.create_dataset("cy_grape_fid_error_by_timeslice", data=cy_grape_fid_error_by_timeslice)
    hf.create_dataset("crab_elapsed_time_by_timeslice", data=crab_elapsed_time_by_timeslice)
    hf.create_dataset("grape_elapsed_time_by_timeslice", data=grape_elapsed_time_by_timeslice)